In [2]:
import pandas as pd

In [4]:
# loading the movie -> director strings, and the new director table with IDs
junction = pd.read_csv("movie_lookup.csv")
director = pd.read_csv("director_extract.csv")

def fix_encoding(s): # fix encoding again/garbled characters
    try:
        return s.encode('latin1').decode('utf-8')
    except (UnicodeEncodeError, UnicodeDecodeError):
        return s

# keeping only movieid + director, dropping movies without director info
director_data = junction[['movieid', 'director']].dropna().copy()

# repairing encoding, then split "A,B" into a list
director_data['director'] = director_data['director'].apply(fix_encoding)
director_data['director'] = director_data['director'].str.split(',')
print(director_data)

       movieid                     director
0            1                   [Don Hahn]
1            2             [Hayao Miyazaki]
2            3                [Jay Russell]
3            4                 [Steve Carr]
4            5       [Francis Ford Coppola]
...        ...                          ...
11359    11360          [Denzel Washington]
11360    11361               [Andrew Erwin]
11361    11362                  [Joel Coen]
11362    11363  [Apichatpong Weerasethakul]
11363    11364              [Clint Bentley]

[11350 rows x 2 columns]


In [7]:
# making it one row per "movie, director" & remove spaces around names
director_data = director_data.explode('director')
director_data['director'] = director_data['director'].str.strip()   
print(director_data)

       movieid                   director
0            1                   Don Hahn
1            2             Hayao Miyazaki
2            3                Jay Russell
3            4                 Steve Carr
4            5       Francis Ford Coppola
...        ...                        ...
11359    11360          Denzel Washington
11360    11361               Andrew Erwin
11361    11362                  Joel Coen
11362    11363  Apichatpong Weerasethakul
11363    11364              Clint Bentley

[11352 rows x 2 columns]


In [9]:
# Removing empty entries and suffix fragments (Jr. etc.)
director_data = director_data[~director_data['director'].isin(['', 'Jr.', 'Sr.', 'II', 'III'])]


In [10]:
# looking up each director's ID: match name in director_data against directorname in the director table
movie_director = director_data.merge(
    director,
    left_on= 'director',
    right_on= 'directorname',
    how='left'
)

In [11]:
# checking the results
movie_director.head(20)

,movieid,director,directorid,directorname
0,1,Don Hahn,1728,Don Hahn
1,2,Hayao Miyazaki,2377,Hayao Miyazaki
2,3,Jay Russell,2726,Jay Russell
3,4,Steve Carr,5571,Steve Carr
4,5,Francis Ford Coppola,2065,Francis Ford Coppola
5,6,Kris Isacsson,3542,Kris Isacsson
6,7,Rodrigo García,5124,Rodrigo García
7,8,Stephan Elliott,5515,Stephan Elliott
8,9,Stewart Raffill,5617,Stewart Raffill
9,10,Andrew Bergman,346,Andrew Bergman


In [12]:
# Counting rows where no ID was found
print(movie_director['directorid'].isna().sum())

0


In [13]:
movie_director.info()
movie_director.isnull().sum().sum()

movie_director = movie_director[['movieid','directorid']].dropna()

movie_director = movie_director.drop_duplicates(
    subset=['movieid', 'directorid']
)
print(movie_director.shape)

<class 'pandas.DataFrame'>
RangeIndex: 11350 entries, 0 to 11349
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   movieid       11350 non-null  int64
 1   director      11350 non-null  str  
 2   directorid    11350 non-null  int64
 3   directorname  11350 non-null  str  
dtypes: int64(2), str(2)
memory usage: 654.1 KB
(11350, 2)


In [14]:
# Export
movie_director = movie_director[['movieid', 'directorid']]
movie_director.to_csv("movie_director.csv", index=False)

In [15]:
# confirm trim happened
movie_director.columns

Index(['movieid', 'directorid'], dtype='str')

In [16]:
# spot check to prove IDs point at right people
check = movie_director.merge(director, on='directorid')
print(check[check['movieid'].isin([1, 7, 3144])])

      movieid  directorid    directorname
0           1        1728        Don Hahn
6           7        5124  Rodrigo García
3141     3144        4210  Michael Landon
